# Load the dataset

## Define the u-net

In [1]:
import os
import sys

# 1. SETUP THE ENVIRONMENT (MUST be first)
os.environ['LD_LIBRARY_PATH'] = '/usr/lib/wsl/lib:' + os.environ.get('LD_LIBRARY_PATH', '')
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True,max_split_size_mb:64'

import torch
import torch.nn as nn
import torch.multiprocessing as mp
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import glob
import cv2
import numpy as np

# 2. PREVENT WORKER COLLISION
try:
    mp.set_start_method('spawn', force=True)
except RuntimeError:
    pass

# 3. VERIFY AND WARM UP
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    # Forces a single allocation to lock the driver
    _ = torch.zeros(1).to("cuda")
    print(f"✅ A6000 Linked: {torch.cuda.get_device_name(0)}")
    print(f"📊 Capacity: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

RuntimeError: CUDA driver error: unknown error

In [ ]:

class SimpleUNet(nn.Module):
    def __init__(self):
        super(SimpleUNet, self).__init__()
        
        # Encoder (Downscaling: Learning "What" is in the image)
        self.enc1 = self.conv_block(3, 64)
        self.enc2 = self.conv_block(64, 128)
        self.pool = nn.MaxPool2d(2)
        
        # Bottleneck (The deepest representation)
        self.bottleneck = self.conv_block(128, 256)
        
        # Decoder (Upscaling: Learning "Where" the mask should be)
        self.up2 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.dec2 = self.conv_block(256, 128) # Concatenation (Skip Connection)
        self.up1 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.dec1 = self.conv_block(128, 64)
        
        # Final Output (1 channel for the probability: 0.0 to 1.0)
        self.final = nn.Conv2d(64, 1, kernel_size=1)

    def conv_block(self, in_ch, out_ch):
        """Standard double-convolution block with Batch Normalization."""
        return nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        # 1. Encoding
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        
        # 2. Bottleneck
        b = self.bottleneck(self.pool(e2))
        
        # 3. Decoding with Skip Connections
        # We 'cat' (concatenate) the upsampled features with the encoder features
        d2 = self.up2(b)
        d2 = self.dec2(torch.cat((d2, e2), dim=1))
        
        d1 = self.up1(d2)
        d1 = self.dec1(torch.cat((d1, e1), dim=1))
        
        # Sigmoid squashes output to a 0-1 range for a probability mask
        return self.final(d1)

print("🧠 SimpleUNet class defined.")

class ProUNet(nn.Module):
    def __init__(self):
        super(ProUNet, self).__init__()
        
        # 4 Levels of depth instead of 2
        self.enc1 = self.conv_block(3, 64)
        self.enc2 = self.conv_block(64, 128)
        self.enc3 = self.conv_block(128, 256)
        self.enc4 = self.conv_block(256, 512)
        self.pool = nn.MaxPool2d(2)
        
        self.bottleneck = self.conv_block(512, 1024)
        
        self.up4 = nn.ConvTranspose2d(1024, 512, 2, 2)
        self.dec4 = self.conv_block(1024, 512)
        self.up3 = nn.ConvTranspose2d(512, 256, 2, 2)
        self.dec3 = self.conv_block(512, 256)
        self.up2 = nn.ConvTranspose2d(256, 128, 2, 2)
        self.dec2 = self.conv_block(256, 128)
        self.up1 = nn.ConvTranspose2d(128, 64, 2, 2)
        self.dec1 = self.conv_block(128, 64)
        
        self.final = nn.Conv2d(64, 1, 1)

    def conv_block(self, in_ch, out_ch):
        return nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))
        
        b = self.bottleneck(self.pool(e4))
        
        d4 = self.up4(b)
        d4 = self.dec4(torch.cat((d4, e4), dim=1))
        d3 = self.up3(d4)
        d3 = self.dec3(torch.cat((d3, e3), dim=1))
        d2 = self.up2(d3)
        d2 = self.dec2(torch.cat((d2, e2), dim=1))
        d1 = self.up1(d2)
        d1 = self.dec1(torch.cat((d1, e1), dim=1))
        
        return self.final(d1)

print("🧠 ProUNet class defined with deeper architecture.")

In [ ]:
class MattingDataset(Dataset):
    def __init__(self, img_dir, msk_dir, target_size=(512, 512)):
        # Collect and sort paths to ensure images and masks match up
        self.img_paths = sorted(glob.glob(os.path.join(img_dir, "*.jpg")))
        self.msk_paths = sorted(glob.glob(os.path.join(msk_dir, "*.png")))
        
        self.transform = transforms.Compose([
            transforms.Resize(target_size),
            transforms.ToTensor(),
        ])

    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, idx):
        # Cleaned strings: No backslashes!
        img = Image.open(self.img_paths[idx]).convert("RGB")
        msk = Image.open(self.msk_paths[idx]).convert("L") # 'L' is for 8-bit Grayscale
        
        return self.transform(img), self.transform(msk)

# --- INITIALIZE THE DATA BRIDGE ---
# Ensure these folder paths match your actual WSL directory structure
base_path = '/home/andrew/CODE/EYEKEY/dataset'
img_dir = os.path.join(base_path, 'train/images')
msk_dir = os.path.join(base_path, 'train/masks')

dataset = MattingDataset(img_dir, msk_dir, target_size=(512, 512))
dataloader = DataLoader(dataset, batch_size=16, shuffle=True, num_workers=0, pin_memory=True)

print(f"📦 Data Bridge Established: Found {len(dataset)} samples ready for training.")

## Training

In [ ]:
# debug test

try:
    test_tensor = torch.zeros((100, 100)).to("cuda")
    print("✅ Smoke test passed: CUDA is responding correctly.")
    del test_tensor
except Exception as e:
    print(f"❌ CUDA still failing: {e}")

In [ ]:
import torch.optim as optim
import random

# 1. Setup Device (Use GPU if you have one in WSL2, otherwise CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SimpleUNet().to(device) # Using the U-Net we defined earlier

# 2. Optimizer and Loss Function
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.BCEWithLogitsLoss() 

print(f"🚀 Training on {device}...")

# 3. The Loop
model.train()
for epoch in range(30):
    epoch_loss = 0
    for images, masks in dataloader:
        # FORCE everything to the A6000 and ensure FLOAT32
        images = images.to(device, dtype=torch.float32)
        masks = masks.to(device, dtype=torch.float32) # BCE needs float masks!
        
        optimizer.zero_grad(set_to_none=True) # More memory efficient than zero_grad()
        
        outputs = model(images)
        
        # Ensure shapes match (B, 1, 512, 512)
        loss = criterion(outputs, masks)
        
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        
        # Optional: cleanup intermediate tensors
        del outputs, loss
    
    print(f"Epoch [{epoch+1}], Loss: {epoch_loss/len(dataloader):.4f}")
    torch.cuda.empty_cache() # Flush after every epoch

print("✨ Training Complete! The brain has finished its first workout.")

## Find the NDI Feeds

In [ ]:
import NDIlib as ndi
import time as time

# 1. Initialize (Safe to run multiple times)
if not ndi.initialize():
    print("❌ NDI failed.")
else:
    # 2. Create finder
    ndi_find = ndi.find_create_v2()
    sources = []
    
    print("📡 Scanning network for 2 seconds...")
    # Give it a moment to hear all the mDNS "shouts"
    time.sleep(2) 
    
    ndi.find_wait_for_sources(ndi_find, 1000)
    sources = ndi.find_get_current_sources(ndi_find)

    if not sources:
        print("❓ No NDI sources found. Check your phone app and Wi-Fi.")
    else:
        print(f"✅ Found {len(sources)} sources:")
        for i, s in enumerate(sources):
            print(f"  [{i}] {s.ndi_name} (IP: {s.url_address})")

# 1. DEFINE THE SETTINGS (The 'ndi_recv_create' you were missing)
ndi_recv_create = ndi.RecvCreateV3()
ndi_recv_create.color_format = ndi.RECV_COLOR_FORMAT_BGRX_BGRA
ndi_recv_create.bandwidth = ndi.RECV_BANDWIDTH_HIGHEST
ndi_recv_create.allow_video_fields = False # Keep it progressive for the AI

# 2. INITIALIZE RECEIVERS
# Clean up old ones first to prevent the kernel crash we talked about
if 'ndi_recv_fill' in globals() and ndi_recv_fill is not None:
    ndi.recv_destroy(ndi_recv_fill)
if 'ndi_recv_key' in globals() and ndi_recv_key is not None:
    ndi.recv_destroy(ndi_recv_key)

# 3. CREATE & CONNECT
# Assumes sources[0] is iPhone and sources[1] is UE5 from your scanner cell
try:
    ndi_recv_fill = ndi.recv_create_v3(ndi_recv_create)
    ndi.recv_connect(ndi_recv_fill, sources[0])
    
    ndi_recv_key = ndi.recv_create_v3(ndi_recv_create)
    ndi.recv_connect(ndi_recv_key, sources[1])
    
    print(f"✅ FILL Connected: {sources[0].ndi_name}")
    print(f"✅ KEY Connected: {sources[1].ndi_name}")
except Exception as e:
    print(f"❌ Connection failed: {e}. Did you run the Scanner cell recently?")

### Run a quick test to see if the masking can be generated when shown the fill only on green.

In [ ]:
import torch
import torchvision.transforms as T
from IPython.display import display, clear_output
import PIL.Image

# Ensure the model is ready
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.eval()

def live_rectify(frame_bgra):
    """Matches the Training logic: Rotate 90 and pad to square."""
    frame = cv2.cvtColor(np.copy(frame_bgra), cv2.COLOR_BGRA2RGB)
    frame = cv2.rotate(frame, cv2.ROTATE_90_CLOCKWISE)
    
    h, w = frame.shape[:2]
    pad = (h - w) // 2
    return cv2.copyMakeBorder(frame, 0, 0, pad, pad, cv2.BORDER_CONSTANT, value=0)

print("🚀 Live AI Inference Active (Vertical Mode).")

try:
    while True:
        # 'v' is the video frame object returned by NDI
        t, v, _, _ = ndi.recv_capture_v3(ndi_recv_fill, 1000)

        if t == ndi.FRAME_TYPE_VIDEO:
            # 1. Rectify the live frame
            frame_rgb = live_rectify(v.data)
            
            # 2. Prep for U-Net
            input_tensor = T.Compose([
                T.ToPILImage(),
                T.Resize((512, 512)), # Match the training input size
                T.ToTensor(),
            ])(frame_rgb).unsqueeze(0).to(device)

            # 3. AI Prediction
            with torch.no_grad():
                pred_mask = model(input_tensor)
            
            # 4. Visualization
            mask_np = pred_mask.squeeze().cpu().numpy()
            binary_mask = (mask_np > 0.5).astype(np.uint8) * 255
            binary_mask_resized = cv2.resize(binary_mask, (frame_rgb.shape[1], frame_rgb.shape[0]))
            result = cv2.bitwise_and(frame_rgb, frame_rgb, mask=binary_mask_resized)

            # Show side-by-side
            preview = np.hstack((cv2.resize(frame_rgb, (360, 360)), 
                                 cv2.resize(result, (360, 360))))
            
            clear_output(wait=True)
            display(PIL.Image.fromarray(preview))

            ndi.recv_free_video_v2(ndi_recv_fill, v)

except KeyboardInterrupt:
    print("⏹️ Inference stopped.")